# Project Sentinel — Exploration

This notebook performs the Phase 2 structural audit, native-loop EDA, and quality verification required by the project brief.

**Important:** Run `src/pipeline.py` first so the raw IDs, generated ground-station log, and processed CSV correspond to the same live API pull.

In [ ]:
from pathlib import Path
import csv
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.pipeline import safe_float, native_stats, quality_audit, validation_crosstab

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'clean_data.csv'
print('Project root:', ROOT)


## 1. Load the processed cohort

In [ ]:
records = []
with PROCESSED_PATH.open('r', encoding='utf-8', newline='') as handle:
    records = list(csv.DictReader(handle))
print('Processed rows:', len(records))
print('Columns:', list(records[0].keys()) if records else [])


## 2. Recursive structural audit

The function below walks dictionaries recursively and inspects the first item of lists, making nested type inconsistencies visible.

In [ ]:
def inspect_leaves(value, path='root'):
    if isinstance(value, dict):
        for key, child in value.items():
            inspect_leaves(child, f'{path}.{key}')
    elif isinstance(value, list):
        if value:
            inspect_leaves(value[0], f'{path}[0]')
        else:
            print(f'{path}: empty list')
    else:
        print(f'{path}: {type(value).__name__}')

# Re-run this cell with 3–5 raw records when raw API JSON is available.
print('The pipeline normalizes raw API values before writing clean_data.csv.')


## 3. Native-loop EDA

The project explicitly forbids `min()`, `max()`, and `sum()` for this exercise. The imported `native_stats()` performs a single-pass min/max/mean calculation.

In [ ]:
fields = [
    'estimated_diameter_max_km',
    'miss_distance_km',
    'relative_velocity_kph',
]
for field in fields:
    values = []
    for row in records:
        values.append({'estimated_diameter_max_km':'estimated_diameter_max_km',
                       'miss_distance_km':'miss_distance_km',
                       'relative_velocity_kph':'relative_velocity_kph'}[field])
    # Native loop directly over CSV rows for the required statistics.
    if records:
        minimum = float(records[0][field])
        maximum = minimum
        running_sum = 0.0
        for row in records:
            value = float(row[field])
            if value < minimum:
                minimum = value
            if value > maximum:
                maximum = value
            running_sum += value
        mean = running_sum / len(records)
        print(f'{field}: min={minimum:.6f}, max={maximum:.6f}, mean={mean:.6f}')


## 4. Quality verification

In [ ]:
def pct_missing(field):
    if not records:
        return 0.0
    missing = 0
    for row in records:
        if row.get(field, '') in ('', None):
            missing += 1
    return missing / len(records) * 100

print('absolute_magnitude_h missing:', f'{pct_missing("absolute_magnitude_h"):.2f}%')
print('confidence_score missing:', f'{pct_missing("confidence_score"):.2f}%')

pha_bad = 0
for row in records:
    if row.get('is_potentially_hazardous_asteroid') not in ('True', 'False'):
        pha_bad += 1
print('NASA PHA non-boolean rows:', pha_bad)


## 5. Validation check

Sentinel's size-plus-distance classifier is intentionally different from NASA's PHA flag, so disagreement is expected.

In [ ]:
table = {
    (False, False): 0,
    (False, True): 0,
    (True, False): 0,
    (True, True): 0,
}
for row in records:
    sentinel = row['priority_watch'] == '1'
    nasa = row['is_potentially_hazardous_asteroid'] == 'True'
    table[(sentinel, nasa)] += 1

print('             NASA=False   NASA=True')
print('Sentinel=False', table[(False, False)], '          ', table[(False, True)])
print('Sentinel=True ', table[(True, False)], '          ', table[(True, True)])


## 6. ROI calculation

The final headline metric is calculated directly from the cleaned dataset:

`pct_workload_reduction = (1 - (n_flagged / n_total)) * 100`


In [ ]:
n_total = len(records)
n_flagged = 0
for row in records:
    if row['priority_watch'] == '1':
        n_flagged += 1
reduction = (1 - (n_flagged / n_total)) * 100 if n_total else 0.0
print(f'An analyst who only manually reviews priority_watch == 1 objects cuts their weekly review set by {reduction:.2f}%.')
